<a href="https://colab.research.google.com/github/kkinca/medical-rag-assistant/blob/main/RAG_Project_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
# Installation for GPU llama-cpp-python

!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 65.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 254.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 323.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 220.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.4 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.


In [ ]:
# ===== CLEAN INSTALL =====

!pip -q install -U pip

# Keep the LangChain family consistent (this matters)
!pip -q install \
  "langchain==0.3.27" \
  "langchain-core==0.3.72" \
  "langchain-community==0.3.27" \
  "langchain-text-splitters==0.3.9"

# Fix the NumPy / compiled-extension mess that causes the `_center` error
!pip -q install \
  "numpy==1.26.4" \
  "scipy==1.11.4" \
  "scikit-learn==1.4.2"

# Sentence Transformers (use a stable version)
!pip -q install "sentence-transformers==2.7.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.0.8 requires langchain-core>=1.0.0, but you have langchain-core 0.3.72 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pointpats 2.5.5 requires scipy>=1.12, but you have scipy 1.11.4 which is incompatible.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
access 1.1.10.post3 requires scipy>=1.14.1, but you have scipy 1.11.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which 

In [ ]:
# # FIX TORCH / TORCHVISION MISMATCH (required for embeddings)

# !pip uninstall -y torchvision torchaudio -q
# !pip install --no-cache-dir torchvision --index-url https://download.pytorch.org/whl/cu121 -q

In [ ]:
#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

In [ ]:
import numpy, packaging
print("numpy:", numpy.__version__)
print("packaging:", packaging.__version__)

numpy: 2.0.2
packaging: 26.0


In [ ]:
#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

## Question Answering using LLM

### Downloading and Loading the model

In [ ]:
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"

In [ ]:
model_path = hf_hub_download(
    repo_id= model_name_or_path,
    filename= model_basename
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


mistral-7b-instruct-v0.2.Q6_K.gguf:   0%|          | 0.00/5.94G [00:00<?, ?B/s]

In [ ]:
llm = Llama(
    model_path=model_path,
    n_ctx=2300,
    n_gpu_layers=38,
    n_batch=512
)

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 1 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


In [ ]:
#llm = Llama(
#    model_path=model_path,
#    n_ctx=1024,
#    n_cores=-2
#)

### Response

In [ ]:
def response(query,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

In [ ]:
response("What treatment options are available for managing hypertension?")

'\n\nHypertension, or high blood pressure, is a common condition that can increase the risk of various health problems such as heart disease, stroke, and kidney damage. The good news is that there are several effective treatment options available to help manage hypertension and reduce the risk of complications. Here are some of the most commonly used treatments:\n\n1. Lifestyle modifications: Making lifestyle changes is often the first line of defense against hypertension. This may include eating a healthy diet rich in fruits, vegetables, whole grains, and lean proteins; limiting sodium intake; getting regular physical activity'

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
response(user_input)

Llama.generate: prefix-match hit


'\n\nSepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are general steps for managing sepsis in a critical care unit:\n\n1. Early recognition: Recognize the signs and symptoms of sepsis early and initiate treatment as soon as possible. Sepsis can present with various clinical features, including fever or hypothermia, tachycardia or bradycardia, altered mental status, respiratory distress, and lactic acidosis.\n2. ABCs'

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
system_prompt = "You are a knowledgeable medical assistant. Provide clear, structured, and professional answers based on medical knowledge. Keep responses informative and concise."

user_input = system_prompt + "\n" + "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

result = llm(
    user_input,
    max_tokens=600,
    temperature=0.2
)

print(result["choices"][0]["text"])

Llama.generate: prefix-match hit



Appendicitis is a medical condition characterized by inflammation of the appendix, a small pouch-like structure located in the lower right side of the abdomen. The common symptoms of appendicitis include:
1. Sudden and persistent pain in the lower right abdomen that may migrate to the center of the abdomen or the back.
2. Loss of appetite, nausea, and vomiting.
3. Fever, which may be low-grade at first but can rise as high as 101°F (38.3°C) or higher.
4. Constipation or diarrhea.
5. Abdominal swelling and rigidity.
6. Inability to pass gas or have a bowel movement.
7. Pain upon walking, coughing, or taking deep breaths.
Appendicitis cannot be cured via medicine alone as the inflammation can lead to perforation of the appendix, which can result in peritonitis, a potentially life-threatening condition. Therefore, surgical intervention is necessary to remove the appendix before it ruptures. The most common surgical procedure for treating appendicitis is an appendectomy, which involves re

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
system_prompt = "You are a knowledgeable medical assistant. Provide clear, structured, and professional answers based on medical knowledge. Keep responses informative and concise."

user_input = system_prompt + "\n" + "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

result = llm(
    user_input,
    max_tokens=900,   # increased for a longer answer
    temperature=0.2
)

print(result["choices"][0]["text"])

Llama.generate: prefix-match hit




Sudden patchy hair loss, also known as alopecia areata, is an autoimmune disorder that results in the sudden loss of hair in small, round patches from the scalp or other areas of the body. The exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system.

Effective treatments for addressing sudden patchy hair loss include:

1. Corticosteroids: These medications can be applied directly to the affected area or taken orally to reduce inflammation and suppress the immune system response that causes hair loss.
2. Immunotherapy: Injections of certain substances, such as diphenylcyclopropenone (DPCP) or squaric acid dibutylester (SADBE), can help stimulate hair regrowth by altering the immune response.
3. Minoxidil: This medication is applied topically to the scalp and can help slow down hair loss and promote new growth in some cases.
4. Hair transplantation: In severe cases of alopecia areata, hair transplantation may be an option to rest

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
system_prompt = "You are a knowledgeable medical assistant. Provide clear, structured, and professional answers based on medical knowledge. Keep responses informative and concise."

user_input = system_prompt + "\n" + "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"

result = llm(
    user_input,
    max_tokens=900,   # longer answer
    temperature=0.2
)

print(result["choices"][0]["text"])

Llama.generate: prefix-match hit


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
system_prompt = "You are a knowledgeable medical assistant. Provide clear, structured, and professional answers based on medical knowledge. Keep responses informative and concise."

user_input = system_prompt + "\n" + "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"

result = llm(
    user_input,
    max_tokens=600,   # <— makes the answer longer
    temperature=0.2
)

print(result["choices"][0]["text"])


Llama.generate: prefix-match hit



A leg fracture sustained during a hiking trip requires prompt medical attention to ensure proper healing and prevent complications. Here are the necessary precautions and treatment steps:
1. Assess the severity of the injury: Check for signs of open wounds, swelling, bruising, deformity, or inability to move the leg. If there is significant pain, swelling, or deformity, do not attempt to move the person unless it is necessary for their safety.
2. Provide first aid: Apply a sterile dressing to any open wounds and apply a splint or immobilize the leg with a makeshift sling or bandages to prevent further movement.
3. Seek medical help: Call emergency services if the injury is severe, or arrange for transportation to a hospital or urgent care center.
4. Pain management: Administer pain medication as prescribed by a healthcare professional to manage pain and discomfort.
5. Immobilization: Keep the leg immobilized using a cast, brace, or splint to prevent further damage and promote healing.

## Question Answering using LLM with Prompt Engineering

In [ ]:
system_prompt = "You are a knowledgeable medical assistant. Provide clear, structured, and professional answers based on medical knowledge. Keep responses informative and concise."

# **Prompt Engineering – Parameter Tuning Experiments**

In [ ]:
import json, re

# ---- 1) Use your existing system prompt from the notebook ----
SYSTEM_PROMPT = system_prompt  # assumes system_prompt is already defined above

# ---- 2) Define 5 parameter combos (rubric requirement) ----
param_runs = [
    {"Run": 1, "temperature": 0.0, "max_tokens": 250, "top_p": 0.90, "top_k": 40},
    {"Run": 2, "temperature": 0.3, "max_tokens": 350, "top_p": 0.95, "top_k": 50},
    {"Run": 3, "temperature": 0.2, "max_tokens": 350, "top_p": 0.90, "top_k": 40},  # likely best
    {"Run": 4, "temperature": 0.6, "max_tokens": 300, "top_p": 0.95, "top_k": 50},
    {"Run": 5, "temperature": 0.2, "max_tokens": 200, "top_p": 0.85, "top_k": 30},
]

# ---- 3) Use one representative question to compare runs ----
TEST_QUERY = "What is the protocol for managing sepsis in a critical care unit?"

def run_once(query: str, params: dict):
    """
    Runs one parameter combo using your existing response() function.
    If your response() signature differs, edit ONLY the response(...) call below.
    """
    user_input_local = SYSTEM_PROMPT + "\n\n" + query
    return response(
        user_input_local,
        temperature=params["temperature"],
        max_tokens=params["max_tokens"],
        top_p=params["top_p"],
        top_k=params["top_k"],
    )

def auto_outcome(text: str) -> str:
    """Simple automatic summary so you don't manually label outcomes."""
    t = text.strip()
    if len(t) < 500:
        length = "Too short"
    elif len(t) > 1500:
        length = "Too verbose"
    else:
        length = "Balanced length"

    structured = "Stepwise" if (re.search(r"\b1\.", t) or "Step" in t or "\n-" in t) else "Less structured"
    cautious = "Cautious" if any(w in t.lower() for w in ["consider", "may", "if feasible", "depending"]) else "More assertive"

    return f"{length}; {structured}; {cautious}"

# ---- 4) Run experiments and store results ----
results = []
for run in param_runs:
    out = run_once(TEST_QUERY, run)
    results.append({
        "Run": run["Run"],
        "Temperature": run["temperature"],
        "MaxTokens": run["max_tokens"],
        "Top-p": run["top_p"],
        "Top-k": run["top_k"],
        "Outcome": auto_outcome(out),
    })

# ---- 5) Print a clean table (copy into Slide 8) ----
headers = ["Run", "Temperature", "MaxTokens", "Top-p", "Top-k", "Outcome"]
col_widths = {h: max(len(h), max(len(str(r[h])) for r in results)) for h in headers}

def print_row(row_dict):
    return " | ".join(str(row_dict[h]).ljust(col_widths[h]) for h in headers)

print("\nPrompt Engineering + Parameter Tuning (5 Runs) — Sepsis Query\n")
print(print_row({h: h for h in headers}))
print("-" * (sum(col_widths.values()) + 3 * (len(headers) - 1)))

for r in results:
    print(print_row(r))

# Optional: save as JSON to reuse later
with open("prompt_tuning_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("\nSaved: prompt_tuning_results.json")

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit



Prompt Engineering + Parameter Tuning (5 Runs) — Sepsis Query

Run | Temperature | MaxTokens | Top-p | Top-k | Outcome                                  
-----------------------------------------------------------------------------------------
1   | 0.0         | 250       | 0.9   | 40    | Balanced length; Stepwise; More assertive
2   | 0.3         | 350       | 0.95  | 50    | Balanced length; Stepwise; Cautious      
3   | 0.2         | 350       | 0.9   | 40    | Too verbose; Stepwise; Cautious          
4   | 0.6         | 300       | 0.95  | 50    | Balanced length; Stepwise; Cautious      
5   | 0.2         | 200       | 0.85  | 30    | Balanced length; Stepwise; More assertive

Saved: prompt_tuning_results.json


### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input = system_prompt+"\n"+ "What is the protocol for managing sepsis in a critical care unit?"
response(user_input)

Llama.generate: prefix-match hit


'\nSepsis is a life-threatening condition caused by a dysregulated response to infection. In a critical care unit, managing sepsis involves a multidisciplinary approach aimed at addressing the underlying infection, supporting organ function, and preventing complications. Here are the key steps in managing sepsis in a critical care unit:\n1. Early recognition and diagnosis: Recognize sepsis early based on clinical signs and laboratory findings such as fever, tachycardia, respiratory distress, lactate level, and inflammatory markers. Use the Sequential Organ Fail'

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input = system_prompt + "\n" + "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

llm_response = llm(
    user_input,
    max_tokens=600, # Increased max_tokens for a more complete response
    temperature=0.2
)

print(llm_response["choices"][0]["text"])

Llama.generate: prefix-match hit



Appendicitis is a medical condition characterized by inflammation of the appendix, a small pouch-like structure that extends from the cecum in the large intestine. The common symptoms of appendicitis include:
1. Abdominal pain, usually starting around the navel area and then shifting to the lower right side of the abdomen.
2. Loss of appetite and feeling sick to your stomach (nausea).
3. Vomiting.
4. Fever and chills.
5. Constipation or diarrhea.
6. Abdominal swelling and tenderness.
7. Inability to pass gas or have a bowel movement.
Appendicitis cannot be cured via medicine alone as the inflammation can lead to appendix rupture, which can result in peritonitis - a serious infection of the abdominal cavity. If left untreated, it can be life-threatening. The standard treatment for appendicitis is surgical removal of the appendix through an appendectomy procedure. This surgery can be performed as an open or laparoscopic procedure, depending on the severity and individual case. After the

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input = system_prompt + "\n" + "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

llm_response = llm(
    user_input,
    max_tokens=600, # Increased max_tokens for a more complete response
    temperature=0.2
)

print(llm_response["choices"][0]["text"])

Llama.generate: prefix-match hit




Sudden patchy hair loss, also known as alopecia areata, is an autoimmune disorder that results in the sudden loss of hair from specific areas of the scalp or other parts of the body. The exact cause of alopecia areata is unknown, but it's believed to be related to a problem with the immune system.

Effective treatments for addressing sudden patchy hair loss include:

1. Corticosteroids: These medications can be applied directly to the affected area or taken orally to reduce inflammation and suppress the immune system response that causes hair loss. Topical corticosteroids, such as minoxidil, are often used in combination with oral corticosteroids for best results.
2. Immunotherapy: Injections of certain substances, such as diphencyprone or squaric acid dibutyl ester, can help stimulate the immune system to promote hair regrowth. This treatment is typically used when other treatments have not been effective.
3. Minoxidil: This medication is applied topically and has been shown to prom

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input = system_prompt + "\n" + "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"

llm_response = llm(
    user_input,
    max_tokens=600, # Increased max_tokens for a more complete response
    temperature=0.2
)

print(llm_response["choices"][0]["text"])

Llama.generate: prefix-match hit




A person with a brain injury may require a combination of medical interventions and rehabilitative therapies to manage their condition and promote recovery. The specific treatments recommended can depend on the severity and location of the injury, as well as the individual's overall health and response to treatment. Here are some common treatments for brain injuries:

1. Emergency care: For severe brain injuries, immediate medical attention is necessary to prevent further damage or complications. This may include surgery to remove hematomas or repair skull fractures, administration of medications to manage swelling or seizures, and supportive measures such as ventilators or feeding tubes if the person is unable to breathe or eat on their own.
2. Medications: Depending on the symptoms and complications of a brain injury, various medications may be prescribed to help manage conditions such as pain, seizures, infections, or depression. For example, anticonvulsants may be used to prevent

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input = system_prompt + "\n" + "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"

llm_response = llm(
    user_input,
    max_tokens=600, # Increased max_tokens for a more complete response
    temperature=0.2
)

print(llm_response["choices"][0]["text"])

Llama.generate: prefix-match hit



A leg fracture sustained during a hiking trip requires prompt medical attention to ensure proper healing and prevent complications. Here are the necessary precautions and treatment steps:
1. Assess the severity of the injury: Determine if it is an open or closed fracture, and assess for any signs of nerve or vessel damage, such as numbness, tingling, or pulselessness.
2. Immobilize the leg: Use a splint or a sling to immobilize the leg above the fracture site to prevent further injury and promote healing.
3. Control bleeding: Apply direct pressure to the wound with a clean cloth to control any bleeding. Elevate the injured leg to reduce swelling and pain.
4. Seek medical help: Call emergency services or arrange for transportation to a hospital if the fracture is severe, open, or accompanied by signs of nerve or vessel damage.
5. Pain management: Use over-the-counter pain relievers such as acetaminophen or ibuprofen to manage pain. Prescription pain medication may be necessary in some 

## Data Preparation for RAG

In [ ]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import SentenceTransformerEmbeddings

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

### Loading the Data

In [ ]:
# uncomment and run the below code snippets if the dataset is present in the Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
manual_pdf_path = "medical_diagnosis_manual.pdf"

In [ ]:
# This cell is now redundant as pdf_loader is created and used in B_17sHnx8oTX
# pdf_loader = PyMuPDFLoader(manual_pdf_path)

In [ ]:
# Install gdown if not already installed
!pip install gdown -q

In [ ]:
# Install pymupdf if not already installed
!pip install pymupdf -q

from langchain_community.document_loaders import PyMuPDFLoader

pdf_loader = PyMuPDFLoader("medical_diagnosis_manual.pdf")
docs = pdf_loader.load()

print("Number of pages loaded:", len(docs))
print(docs[0].page_content[:500])

In [ ]:
# Install gdown if needed
!pip -q install gdown

import gdown
import re

# Google Drive full URL
gdrive_full_url = "https://drive.google.com/file/d/1ilXxg8XLeVvZc3qp29Xxpu0Dm689kxrr/view?usp=sharing"

# Extract the file ID using a regular expression
match = re.search(r'/d/([a-zA-Z0-9_-]+)', gdrive_full_url)
if match:
    file_id = match.group(1)
else:
    raise ValueError("Could not extract file ID from the Google Drive URL.")

# Output filename
local_pdf_filename = "medical_diagnosis_manual.pdf"

# Download using fuzzy=True (more reliable)
url = f"https://drive.google.com/uc?id={file_id}"
gdown.download(url, local_pdf_filename, quiet=False)

print("Download complete!")

In [ ]:
manual_pdf_path = 'medical_diagnosis_manual.pdf' # Updated to point to the locally downloaded file

In [ ]:
manual = docs

### Data Overview

#### Checking the first 5 pages

In [ ]:
for i in range(5):
    print(f"Page Number : {i+1}",end="\n")
    print(manual[i].page_content,end="\n")

#### Checking the number of pages

In [ ]:
len(manual)

### Data Chunking

In [ ]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=1000, #Complete the code to define the chunk size
    chunk_overlap=200 #Complete the code to define the chunk overlap
)

In [ ]:
document_chunks = pdf_loader.load_and_split(text_splitter)

In [ ]:
len(document_chunks)

In [ ]:
document_chunks[0].page_content

In [ ]:
document_chunks[2].page_content

In [ ]:
document_chunks[3].page_content

As expected, there are some overlaps

### Embedding

In [ ]:
embedding_model = SentenceTransformerEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


In [ ]:
vec = embedding_model.embed_query("hello world")
print(len(vec))

In [ ]:
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

In [ ]:
print("Dimension of the embedding vector ",len(embedding_1))
len(embedding_1)==len(embedding_2)

In [ ]:
embedding_1,embedding_2

### Vector Database

In [ ]:
out_dir = 'medical_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

In [ ]:
# Install chromadb if not already installed
!pip install chromadb -q

vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)

In [ ]:
vectorstore.embeddings

In [ ]:
vectorstore.similarity_search("What are the common symptoms and treatments for pulmonary embolism?",k=3) #Complete the code to pass a query and an appropriate k value

### Retriever

In [ ]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3} #Complete the code to pass an appropriate k value
)

In [ ]:
rel_docs = retriever.get_relevant_documents("What are the common symptoms and treatments for pulmonary embolism?") #Complete the code to pass the query
rel_docs

In [ ]:
model_output = llm(
      "What are the common symptoms and treatments for pulmonary embolism?", #Complete the code to pass the query
      max_tokens=256, #Complete the code to pass the maximum number of tokens
      temperature=0.2, #Complete the code to pass the temperature
    )

In [ ]:
model_output['choices'][0]['text']

The above response is somewhat generic and is solely based on the data the model was trained on, rather than the medical manual.  

Let's now provide our own context.

### System and User Prompt Template

Prompts guide the model to generate accurate responses. Here, we define two parts:

    1. The system message describing the assistant's role.
    2. A user message template including context and the question.

In [ ]:
qna_system_message = """
You are a careful and evidence-based medical assistant.

Answer the user's question using ONLY the provided context.
If the answer is not contained in the context, say:
"I do not have enough information in the provided context to answer this question."

Do not fabricate medical information.
Be clear, concise, and structured.
"""

In [ ]:
qna_user_message_template = """
### Context:
{context}

### Question:
{question}

### Answer:
"""

### Response Function

In [ ]:
def generate_rag_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

In [ ]:
# RAG tuning experiments using existing retriever & response function

TEST_QUERY = "What is the protocol for managing sepsis in a critical care unit?"

rag_runs = [
    {"Run": 1, "k": 1, "temperature": 0.2, "max_tokens": 300},
    {"Run": 2, "k": 3, "temperature": 0.2, "max_tokens": 350},  # likely best
    {"Run": 3, "k": 5, "temperature": 0.2, "max_tokens": 350},
    {"Run": 4, "k": 3, "temperature": 0.4, "max_tokens": 350},
    {"Run": 5, "k": 3, "temperature": 0.2, "max_tokens": 250},
]

results = []

for run in rag_runs:

    # Update retriever depth
    retriever.search_kwargs["k"] = run["k"]

    # Generate RAG response using your existing function
    response_text = generate_rag_response(
        user_input=TEST_QUERY,
        temperature=run["temperature"],
        max_tokens=run["max_tokens"]
    )

    length = len(response_text)

    if length < 600:
        observation = "Short; possible missing details"
    elif length > 1500:
        observation = "Verbose; possible redundancy"
    else:
        observation = "Balanced length"

    results.append({
        "Run": run["Run"],
        "k": run["k"],
        "Temperature": run["temperature"],
        "Max Tokens": run["max_tokens"],
        "Observation": observation
    })

print("\nRAG Tuning Experiments (Sepsis Query)\n")
for r in results:
    print(r)

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
generate_rag_response(user_input, k=1, max_tokens=600)

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
generate_rag_response(user_input_2, k=1, max_tokens=600)

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?" #Complete the code to pass the query #3
generate_rag_response(user_input_3, k=1, max_tokens=600) # Reduced k to avoid exceeding context window

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?" #Complete the code to pass the query #4
generate_rag_response(user_input_4, k=1, max_tokens=600) #Complete the code to pass the user input

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?" #Complete the code to pass the query #5
generate_rag_response(user_input_5, k=1, max_tokens=600) # Reduced k to avoid exceeding context window

### Fine-tuning

#### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
generate_rag_response(user_input, temperature=0.5, k=1, max_tokens=600)

#### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?" #Complete the code to pass the query #2
generate_rag_response(user_input_2, temperature=0.5, k=1, max_tokens=600)

#### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?" #Complete the code to pass the query #3
generate_rag_response(user_input_3, k=1, temperature=0.5, max_tokens=600)

#### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?" #Complete the code to pass the query #4
generate_rag_response(user_input_4, temperature=0.5, max_tokens=600)

#### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?" #Complete the code to pass the query #5
generate_rag_response(user_input_5, k=1, temperature=0.5, max_tokens=600)

## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [ ]:
groundedness_rater_system_message = """
You are a strict medical QA evaluator for a Retrieval-Augmented Generation (RAG) system.
Your job is to judge GROUNDEDNESS only.

Definition:
- Groundedness = how well the answer is supported by the provided context.
- If the answer contains claims that are not present in the context, groundedness drops.
- If the answer contradicts the context, groundedness is very low.

Return ONLY a JSON object with exactly these keys:
{
  "score": <integer 1-5>,
  "verdict": "<one of: grounded | partially_grounded | not_grounded>",
  "rationale": "<1-3 sentences, cite phrases from the context when possible>"
}

Scoring:
5 = Fully supported by context, no extra claims.
4 = Mostly supported, minor extrapolation.
3 = Some support, but multiple unsupported additions.
2 = Little support, mostly hallucinated.
1 = Not supported or contradicts context.
"""

In [ ]:
relevance_rater_system_message = """
You are a strict medical QA evaluator for a Retrieval-Augmented Generation (RAG) system.
Your job is to judge RELEVANCE only.

Definition:
- Relevance = whether the answer directly addresses the question asked.
- Do NOT judge truthfulness unless it affects relevance.
- An answer can be relevant but not grounded; ignore grounding here.

Return ONLY a JSON object with exactly these keys:
{
  "score": <integer 1-5>,
  "verdict": "<one of: relevant | partially_relevant | not_relevant>",
  "rationale": "<1-3 sentences explaining why it addresses (or fails to address) the question>"
}

Scoring:
5 = Fully answers the question clearly and specifically.
4 = Answers most of it; slightly generic or missing a small part.
3 = Partially answers; significant gaps.
2 = Barely answers; mostly off-target.
1 = Does not address the question.
"""

In [ ]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [ ]:
import tiktoken

def generate_ground_relevance_response(user_input, k_eval=1, max_tokens=150, temperature=0, top_p=0.95, top_k=50):
    global qna_system_message, qna_user_message_template, groundedness_rater_system_message, user_message_template, relevance_rater_system_message

    tokenizer = tiktoken.get_encoding("cl100k_base")

    # Retrieve relevant document chunks for evaluation
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input, k=k_eval)
    context_list = [d.page_content for d in relevant_document_chunks]
    full_context_for_query = ". ".join(context_list)

    # Truncate context_for_query if it's too long
    # Max tokens for context (approx.) to leave space for system messages, question, and answer
    max_context_tokens_allowed = llm.n_ctx() - 60 - 20 - 30 - 600 - 50 # n_ctx - system - template - question - max_answer - buffer
    if max_context_tokens_allowed < 100: # Ensure at least some context if calculation is too low
        max_context_tokens_allowed = 100

    encoded_context = tokenizer.encode(full_context_for_query)
    if len(encoded_context) > max_context_tokens_allowed:
        truncated_encoded_context = encoded_context[:max_context_tokens_allowed]
        context_for_query = tokenizer.decode(truncated_encoded_context)
        print(f"Warning: Context truncated from {len(encoded_context)} to {max_context_tokens_allowed} tokens for query: {user_input[:50]}...")
    else:
        context_for_query = full_context_for_query

    user_message_for_rag = qna_user_message_template.replace('{context}', context_for_query)
    user_message_for_rag = user_message_for_rag.replace('{question}', user_input)
    prompt_for_rag = qna_system_message + '\n' + user_message_for_rag

    max_tokens_for_answer_generation = 600 # We want detailed answers for RAG generation

    rag_prompt_tokens = len(tokenizer.encode(prompt_for_rag))
    print(f"RAG prompt tokens (before answer generation): {rag_prompt_tokens}")
    print(f"Max tokens for RAG answer generation: {max_tokens_for_answer_generation}")
    print(f"Estimated total for RAG LLM call: {rag_prompt_tokens + max_tokens_for_answer_generation}")

    current_max_tokens_for_rag = max_tokens_for_answer_generation
    if rag_prompt_tokens + current_max_tokens_for_rag > llm.n_ctx():
        current_max_tokens_for_rag = llm.n_ctx() - rag_prompt_tokens - 50 # Leave some buffer
        if current_max_tokens_for_rag < 50: # Ensure at least some generation is possible
            current_max_tokens_for_rag = 50
        print(f"RAG answer generation adjusted max_tokens to: {current_max_tokens_for_rag} due to context window.")

    response_rag = llm(
              prompt=prompt_for_rag,
              max_tokens=current_max_tokens_for_rag,
              temperature=temperature,
              top_p=top_p,
              top_k=top_k,
              stop=['INST'],
              echo=False
              )
    answer = response_rag['choices'][0]['text'].strip()

    # Groundedness evaluation prompt
    groundedness_user_msg = user_message_template.format(context=context_for_query, question=user_input, answer=answer)
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {groundedness_user_msg}
                [/INST]"""

    groundedness_prompt_tokens = len(tokenizer.encode(groundedness_prompt))
    print(f"Groundedness evaluator prompt tokens (before eval generation): {groundedness_prompt_tokens}")
    print(f"Max tokens for groundedness eval generation: {max_tokens}")

    current_max_tokens_for_eval = max_tokens
    if groundedness_prompt_tokens + current_max_tokens_for_eval > llm.n_ctx():
        current_max_tokens_for_eval = llm.n_ctx() - groundedness_prompt_tokens - 50 # Leave some buffer
        if current_max_tokens_for_eval < 50:
            current_max_tokens_for_eval = 50
        print(f"Groundedness evaluation adjusted max_tokens to: {current_max_tokens_for_eval} due to context window.")

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=current_max_tokens_for_eval,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            echo=False
            )

    # Relevance evaluation prompt
    relevance_user_msg = user_message_template.format(context=context_for_query, question=user_input, answer=answer)
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {relevance_user_msg}
                [/INST]"""

    relevance_prompt_tokens = len(tokenizer.encode(relevance_prompt))
    print(f"Relevance evaluator prompt tokens (before eval generation): {relevance_prompt_tokens}")
    print(f"Max tokens for relevance eval generation: {max_tokens}")

    current_max_tokens_for_eval = max_tokens
    if relevance_prompt_tokens + current_max_tokens_for_eval > llm.n_ctx():
        current_max_tokens_for_eval = llm.n_ctx() - relevance_prompt_tokens - 50 # Leave some buffer
        if current_max_tokens_for_eval < 50:
            current_max_tokens_for_eval = 50
        print(f"Relevance evaluation adjusted max_tokens to: {current_max_tokens_for_eval} due to context window.")

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=current_max_tokens_for_eval,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            echo=False
            )

    return response_1['choices'][0]['text'], response_2['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
ground,rel = generate_ground_relevance_response(user_input=user_input, max_tokens=150, k_eval=1)

print(ground,end="\n\n")
print(rel)

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
ground,rel = generate_ground_relevance_response(user_input=user_input_2, max_tokens=150, k_eval=1)

print(ground,end="\n\n")
print(rel)

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
ground,rel = generate_ground_relevance_response(user_input=user_input_3, max_tokens=150, k_eval=1)

print(ground,end="\n\n")
print(rel)

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
ground,rel = generate_ground_relevance_response(user_input=user_input_4, max_tokens=150, k_eval=1)

print(ground,end="\n\n")
print(rel)

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
ground,rel = generate_ground_relevance_response(user_input=user_input_5, max_tokens=150, k_eval=1)

print(ground,end="\n\n")
print(rel)

# New Section

In [ ]:
# === AUTO-BUILD EVALUATION TABLE (Groundedness + Relevance) ===
# Paste this cell right AFTER your Query #5 evaluation cell in the Output Evaluation section.

import json, re
import pandas as pd

def extract_eval_fields(raw: str):
    """
    Robustly extracts score/verdict/rationale from evaluator output.
    1) Try strict JSON parsing
    2) If JSON is malformed (common with unescaped quotes), fall back to regex extraction
    """
    if raw is None:
        return {"score": None, "verdict": None, "rationale": ""}

    s = raw.strip()

    # Try to isolate the JSON object if extra text surrounds it
    if not (s.startswith("{") and s.endswith("}")):
        start = s.find("{")
        end = s.rfind("}")
        if start != -1 and end != -1 and end > start:
            s = s[start:end+1]

    # 1) Strict JSON parse
    try:
        obj = json.loads(s)
        return {
            "score": obj.get("score"),
            "verdict": obj.get("verdict"),
            "rationale": obj.get("rationale", "")
        }
    except Exception:
        # 2) Regex fallback for malformed JSON-ish output
        score = None
        verdict = None
        rationale = ""

        m_score = re.search(r'"score"\s*:\s*(\d+)', s)
        if m_score:
            score = int(m_score.group(1))

        m_verdict = re.search(r'"verdict"\s*:\s*"([^"]+)"', s)
        if m_verdict:
            verdict = m_verdict.group(1)

        m_rat = re.search(r'"rationale"\s*:\s*"(.+?)"\s*(,|\})', s, flags=re.S)
        if m_rat:
            rationale = m_rat.group(1)

        # Extra fallback if model outputs "Score: 4" style (non-JSON)
        if score is None:
            m_score2 = re.search(r"score\s*[:=]\s*(\d+)", raw, flags=re.I)
            if m_score2:
                score = int(m_score2.group(1))

        if verdict is None:
            m_verdict2 = re.search(r"verdict\s*[:=]\s*([A-Za-z \-_]+)", raw, flags=re.I)
            if m_verdict2:
                verdict = m_verdict2.group(1).strip()

        return {"score": score, "verdict": verdict, "rationale": rationale}

queries = [
    ("Query 1 – Sepsis Management", user_input),
    ("Query 2 – Appendicitis", user_input_2),
    ("Query 3 – Patchy Hair Loss", user_input_3),
    ("Query 4 – Traumatic Brain Injury", user_input_4),
    ("Query 5 – Leg Fracture", user_input_5),
]

rows = []

for label, q in queries:
    ground_str, rel_str = generate_ground_relevance_response(
        user_input=q,
        k_eval=1,
        max_tokens=150,
        temperature=0,
        top_p=0.95,
        top_k=50
    )

    ground = extract_eval_fields(ground_str)
    rel = extract_eval_fields(rel_str)

    rows.append({
        "Query": label,
        "Groundedness Score (1–5)": ground.get("score"),
        "Groundedness Verdict": ground.get("verdict"),
        "Relevance Score (1–5)": rel.get("score"),
        "Relevance Verdict": rel.get("verdict"),
        "Groundedness Rationale (short)": (ground.get("rationale") or "")[:120],
        "Relevance Rationale (short)": (rel.get("rationale") or "")[:120],
    })

df_eval = pd.DataFrame(rows)

print("✅ Slide-friendly table (scores only):")
display(df_eval[["Query", "Groundedness Score (1–5)", "Relevance Score (1–5)"]])

print("\n✅ Full table (includes verdict + short rationale):")
display(df_eval)